#Initialization

In [0]:
import pyspark.sql.functions as F

#Read Bronze Table

In [0]:
billing_2021 = spark.table("health.bronze.billing_2021")
billing_2022 = spark.table("health.bronze.billing_2022")
billing_2023 = spark.table("health.bronze.billing_2023")
billing_2024 = spark.table("health.bronze.billing_2024")
billing_2025 = spark.table("health.bronze.billing_2025")

#Data Understanding

###Review Schema

In [0]:
billing_2021.printSchema()
display(billing_2021.limit(10))

###Consolidation

In [0]:
billing = (
    billing_2021
    .unionByName(billing_2022)
    .unionByName(billing_2023)
    .unionByName(billing_2024)
    .unionByName(billing_2025)
)

###Check Schema Again

In [0]:
billing.printSchema()

#Data Profiling

###Check Null Values

In [0]:
display(
    billing.select(
        [
            F.sum(
                F.when(F.col(c).isNull(), 1).otherwise(0)
            ).alias(c)
            for c in billing.columns
        ]
    )
)

###Business Key Validation

In [0]:
total_rows = billing.count()

distinct_billing_ids = (
    billing
    .select("billing_id")
    .distinct()
    .count()
)

print(f"Total Rows: {total_rows}")
print(f"Distinct Billing IDs: {distinct_billing_ids}")

###Check Minimum and Maximum Values

In [0]:
display(
    billing.select(
        F.min("total_cost").alias("min_total_cost"),
        F.max("total_cost").alias("max_total_cost"),
        F.min("insurance_paid").alias("min_insurance_paid"),
        F.max("insurance_paid").alias("max_insurance_paid"),
        F.min("patient_paid").alias("min_patient_paid"),
        F.max("patient_paid").alias("max_patient_paid")
    )
)

###Financial Integrity Validation

In [0]:
billing_mismatch = billing.filter(
    F.round(
        F.col("insurance_paid") + F.col("patient_paid"),
        2
    ) != F.round(F.col("total_cost"), 2)
)

print(
    f"Mismatched Records: {billing_mismatch.count()}"
)

#Transformations

###Round Financial Values to 2 Decimal Places

In [0]:
billing_clean = (
    billing
    .withColumn(
        "insurance_paid",
        F.round(F.col("insurance_paid"), 2)
    )
    .withColumn(
        "patient_paid",
        F.round(F.col("patient_paid"), 2)
    )
)

###Validation Summary

In [0]:
print(f"Clean Billing Records: {billing_clean.count()}")
print(f"Original Billing Records: {billing.count()}")

#Write Silver Tables

###Save Clean Billing


In [0]:
(
    billing_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("health.silver.billing")
)

###Verify Silver Billing

In [0]:
display(
    spark.table("health.silver.billing")
    .limit(10)
)